# NB02 — Python Multiprocessing and Threading

**Module 17: HPC, Parallel Computing, and Rust**

---

## 1. Motivation

Your laptop has 8 cores. NumPy uses one of them for most operations. When a computation is truly sequential — a loop that must finish step N before step N+1 — there is no escape. But when you have 1000 independent tasks (1000 sequences to analyze, 1000 files to process), you can hand one task to each core and finish in 1/8th the time.

Python provides two mechanisms: **threads** and **processes**. Understanding why threads often don't help for CPU-bound work — and when they do help — is the first thing to get right.

---

## 2. Intuition

**The GIL (Global Interpreter Lock):** CPython, the standard Python interpreter, holds a lock that only one thread can hold at a time. This means that even if you launch 8 threads, only one of them can execute Python bytecode at any moment. For CPU-bound work (doing computation), threads don't help — they take turns rather than run in parallel.

**Processes bypass the GIL:** Each process has its own Python interpreter and memory space. They truly run in parallel on separate cores. The cost is higher startup overhead and the need to serialize data (via pickle) to pass it between processes.

**When threads DO help:** IO-bound work (reading files, making network requests, waiting for a database). While one thread is waiting for disk IO, the GIL is released and another thread can run. Threads are the right tool here.

---

## 3. Biological background

**CPU-bound bioinformatics tasks** (use processes):
- GC content computation over thousands of sequences
- Pairwise alignment scoring
- K-mer counting over a large dataset
- Statistical testing across thousands of genes

**IO-bound bioinformatics tasks** (use threads or async):
- Reading FASTA files from disk
- Downloading sequences from NCBI/EBI APIs
- Writing results to multiple output files

Real pipelines often chain both: threads for file IO, then processes for computation.

---

## 4. Mathematical explanation — Amdahl's Law

Let $s$ be the fraction of the program that is **serial** (cannot be parallelized). The theoretical maximum speedup with $n$ cores is:

$$S(n) = \frac{1}{s + \frac{1-s}{n}}$$

Key implications:
- If $s = 0.05$ (5% serial): max speedup = 1/0.05 = **20x**, regardless of cores
- If $s = 0.20$ (20% serial): max speedup = 1/0.20 = **5x**
- Adding more cores helps less and less as the serial fraction dominates

This is why parallelism is not a replacement for algorithmic improvement — if the bottleneck is sequential, Amdahl's law caps your gain.

---

## 5. Computational explanation

### `multiprocessing.Pool`

```python
from multiprocessing import Pool

with Pool(n_workers) as pool:
    results = pool.map(function, iterable)       # one argument
    results = pool.starmap(function, arg_tuples) # multiple arguments
```

- `pool.map`: applies `function` to each element of `iterable` in parallel
- `pool.starmap`: same, but each element of `arg_tuples` is unpacked as `*args`
- Data is pickled (serialized) to send to worker processes — avoid sending large arrays this way

### `concurrent.futures`

```python
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

with ProcessPoolExecutor(max_workers=4) as ex:
    results = list(ex.map(function, iterable))
```

Higher-level API; same semantics. Prefer this for new code — cleaner cancellation and exception handling.

---

## 6. Python implementation

In [ ]:
import numpy as np
import time
import multiprocessing
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import matplotlib.pyplot as plt

np.random.seed(42)
N_CPU = multiprocessing.cpu_count()
print(f"Available CPUs: {N_CPU}")

In [ ]:
# --- Data: 1000 sequences ---
def generate_sequences(N, L, seed=42):
    rng = np.random.default_rng(seed)
    return rng.integers(0, 4, size=(N, L), dtype=np.int8)

seqs = generate_sequences(N=1000, L=500)
# Convert to list of arrays for multiprocessing (avoids shared memory complications)
seqs_list = [seqs[i] for i in range(len(seqs))]
print(f"Dataset: {len(seqs_list)} sequences of length {seqs_list[0].shape[0]}")

In [ ]:
# --- The worker function (must be defined at module level for pickling) ---
def gc_content_single(seq):
    """GC content of a single integer-encoded sequence."""
    return float(((seq == 2) | (seq == 3)).mean())

def gc_content_serial(seqs_list):
    """Serial baseline."""
    return [gc_content_single(s) for s in seqs_list]

# Test serial
t0 = time.perf_counter()
gc_serial = gc_content_serial(seqs_list)
t_serial = time.perf_counter() - t0
print(f"Serial: {t_serial:.4f} s  |  Mean GC: {np.mean(gc_serial):.4f}")

In [ ]:
# --- Pool.map parallel version ---
def gc_parallel_pool(seqs_list, n_workers):
    with Pool(processes=n_workers) as pool:
        return pool.map(gc_content_single, seqs_list)

# Timing across worker counts
worker_counts = [1, 2, 4, min(8, N_CPU)]
times_pool = []

for n in worker_counts:
    # Warmup
    gc_parallel_pool(seqs_list[:10], n)
    
    t0 = time.perf_counter()
    gc_par = gc_parallel_pool(seqs_list, n)
    t_par = time.perf_counter() - t0
    times_pool.append(t_par)
    
    assert np.allclose(gc_serial, gc_par, atol=1e-6), "Results disagree!"
    print(f"Workers={n}: {t_par:.4f} s  |  Speedup: {t_serial/t_par:.2f}x")

In [ ]:
# --- ProcessPoolExecutor (modern API) ---
def gc_with_executor(seqs_list, n_workers):
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        return list(ex.map(gc_content_single, seqs_list, chunksize=50))

# Note: chunksize=50 means each worker receives 50 tasks at a time
# This reduces IPC overhead for small tasks
t0 = time.perf_counter()
gc_executor = gc_with_executor(seqs_list, n_workers=4)
t_ex = time.perf_counter() - t0
assert np.allclose(gc_serial, gc_executor, atol=1e-6)
print(f"ProcessPoolExecutor (4 workers, chunksize=50): {t_ex:.4f} s")

In [ ]:
# --- ThreadPoolExecutor for IO-bound simulation ---
# Simulate reading a FASTA file: each 'task' sleeps for a disk IO delay
import random

def simulate_fasta_read(i):
    """Simulates IO-bound work: a small sleep + some computation."""
    time.sleep(0.001)  # 1ms IO wait
    return i * 2  # trivial CPU work

n_files = 200

# Serial
t0 = time.perf_counter()
r_serial = [simulate_fasta_read(i) for i in range(n_files)]
t_io_serial = time.perf_counter() - t0

# Threads (GIL released during sleep, so threads help here)
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as ex:
    r_thread = list(ex.map(simulate_fasta_read, range(n_files)))
t_io_thread = time.perf_counter() - t0

# Processes (overkill for IO-bound, but let's see)
t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as ex:
    r_proc = list(ex.map(simulate_fasta_read, range(n_files)))
t_io_proc = time.perf_counter() - t0

print(f"IO-bound simulation ({n_files} tasks, 1ms each):")
print(f"  Serial: {t_io_serial:.3f} s")
print(f"  8 Threads: {t_io_thread:.3f} s  (speedup: {t_io_serial/t_io_thread:.1f}x)")
print(f"  4 Processes: {t_io_proc:.3f} s  (speedup: {t_io_serial/t_io_proc:.1f}x)")

In [ ]:
# --- Amdahl's Law ---
def amdahl_speedup(serial_fraction, n_cores):
    """Theoretical maximum speedup given serial fraction s and n cores."""
    s = serial_fraction
    return 1.0 / (s + (1 - s) / n_cores)

cores = np.arange(1, 33)
serial_fractions = [0.0, 0.05, 0.1, 0.2, 0.5]
print("Amdahl's Law — theoretical max speedup:")
print(f"{'s':>6} {'n=2':>8} {'n=4':>8} {'n=8':>8} {'n=16':>8} {'n→∞':>8}")
for s in serial_fractions:
    row = [f"{amdahl_speedup(s, n):.1f}x" for n in [2, 4, 8, 16]]
    lim = f"{1/s:.1f}x" if s > 0 else "∞"
    print(f"{s:>6.2f} {'  '.join(row):>40} {lim:>8}")

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Speedup vs n_workers (empirical) + Amdahl overlay
ax = axes[0]
empirical_speedups = [t_serial / t for t in times_pool]
ax.plot(worker_counts, empirical_speedups, 'o-', color='#2980b9',
        linewidth=2, markersize=8, label='Empirical speedup')

cores_range = np.linspace(1, max(worker_counts), 100)
for s, color, ls in zip([0.0, 0.1, 0.3],
                          ['#27ae60', '#f39c12', '#e74c3c'],
                          ['-', '--', ':']):
    speedups = [amdahl_speedup(s, n) for n in cores_range]
    ax.plot(cores_range, speedups, color=color, linestyle=ls,
            alpha=0.7, label=f"Amdahl s={s:.0%}")

ax.set_xlabel('Number of workers', fontsize=11)
ax.set_ylabel('Speedup over serial', fontsize=11)
ax.set_title('CPU-bound: Speedup vs Workers\n(GC content, N=1000)', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.5, max(worker_counts) + 0.5)
ax.set_ylim(0, max(empirical_speedups) * 1.5)

# Panel 2: CPU-bound vs IO-bound comparison
ax = axes[1]
categories = ['CPU-bound\n(4 processes)', 'IO-bound\n(8 threads)', 'IO-bound\n(4 processes)']
speedups_comp = [
    t_serial / gc_parallel_pool.__wrapped__(seqs_list, 4) if hasattr(gc_parallel_pool, '__wrapped__') else t_serial / times_pool[worker_counts.index(4)],
    t_io_serial / t_io_thread,
    t_io_serial / t_io_proc
]
# Recompute cleanly
speedups_comp = [
    t_serial / times_pool[worker_counts.index(4)],
    t_io_serial / t_io_thread,
    t_io_serial / t_io_proc
]
colors = ['#2980b9', '#27ae60', '#e67e22']
bars = ax.bar(categories, speedups_comp, color=colors, edgecolor='black', alpha=0.85)
for bar, sp in zip(bars, speedups_comp):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{sp:.1f}x', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Speedup over serial', fontsize=11)
ax.set_title('CPU-bound vs IO-bound:\nThreads vs Processes', fontweight='bold')
ax.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='No speedup')
ax.grid(True, axis='y', alpha=0.3)
ax.legend()

plt.tight_layout()
plt.savefig('../datasets/nb02_multiprocessing.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Exercises

1. **Pool.starmap:** Write a function `pairwise_gc_diff(seq_a, seq_b)` that returns `|gc(a) - gc(b)|`. Use `Pool.starmap` to compute this for all pairs in a list of 50 sequences. Compare wall time to a serial implementation.

2. **Amdahl's law calculation:** A bioinformatics pipeline has the following step times: file loading (IO, 2s), quality trimming (CPU, 8s), alignment (CPU, 20s), counting (CPU, 5s), reporting (serial, 1s). What is the theoretical max speedup with 8 cores if you parallelize all CPU steps but not IO/serial steps?

3. **Chunk size effect:** Re-run `gc_parallel_pool` with `Pool.map(gc_content_single, seqs_list, chunksize=1)` vs `chunksize=100` vs `chunksize=500`. How does chunksize affect wall time and why?

4. **ProcessPoolExecutor with exceptions:** Write a version of `gc_content_single` that raises a `ValueError` if the sequence contains any value > 3. Use `ProcessPoolExecutor` and catch the exception gracefully for each failed sequence without stopping the whole computation.

## 12. Reflection

*Write here: In your own words, why does the GIL exist? What would happen if Python removed it? When would you choose threads over processes in a bioinformatics context?*

---

## Papers referenced

- Amdahl, G.M. (1967). "Validity of the single processor approach to achieving large scale computing capabilities." *AFIPS Spring Joint Computer Conference*.

## References

- Python `multiprocessing` docs: https://docs.python.org/3/library/multiprocessing.html
- `concurrent.futures` docs: https://docs.python.org/3/library/concurrent.futures.html
- PEP 703 — Making the GIL optional in CPython (Python 3.13+): https://peps.python.org/pep-0703/

## Future work / open questions

- Python 3.13 introduces a free-threaded build where the GIL is optional. How would that change the CPU-bound threading story?
- For the pairwise Hamming distance matrix (N=10,000), could you parallelize the row computation with Pool.map? What are the memory implications?